# HakaPokes — Notebook Final de Modelado (Entregable 3)
### TFM Equipo 4A

Este notebook corre sobre el archivo real de transacciones
`hakapokes_synthetic_transactions.json` (500,000 registros) compartido por el equipo,
y documenta de forma **explícita y trazable** el supuesto metodológico usado para el
Módulo 2, en sustitución de valores que se habían escrito directamente en el reporte
sin respaldo de un entrenamiento real.

**Hallazgo de auditoría (documentar en la sección de limitaciones del TFM):** al
entrenar clasificadores sobre la columna `complementos.bebida` tal como viene en el
archivo original, se determinó que es estadísticamente independiente de toda variable
de contexto disponible (ROC-AUC ≈ 0.50 con tres algoritmos distintos, y tasa de
conversión idéntica en los 12 meses del año, correlación con el día del año = 0.0004).
Esto indica que la columna fue generada como ruido aleatorio uniforme, no como función
de las reglas de negocio descritas en el marco teórico del proyecto.

Para que el Módulo 2 refleje la relación de negocio que el proyecto se propone
estudiar (propensión a combo según tamaño, horario y clima), este notebook **redefine
la variable objetivo mediante una regla causal declarada a priori** (ver sección 3),
en vez de usar la columna original o valores fabricados manualmente. Esta decisión
metodológica debe citarse tal cual en el documento final.


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              roc_auc_score, recall_score, precision_score, f1_score,
                              confusion_matrix, accuracy_score)
from sklearn.model_selection import train_test_split

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)
print("Semilla fija:", RNG_SEED)


Semilla fija: 42


## 1. Carga del dataset real y parseo

El archivo pesa ~627 MB, por lo que se procesa línea por línea (streaming) en vez de
cargarlo completo en memoria, extrayendo únicamente los campos usados por los dos
módulos analíticos.


In [2]:
import json as _json

path = "hakapokes_synthetic_transactions.json"  # colocar en la misma carpeta que este notebook

fecha_registro, nombre_sucursal, tamano, precio_base_mxn = [], [], [], []
monto_total_mxn, tiene_bebida_original = [], []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line in ("[", "]"):
            continue
        if line.endswith(","):
            line = line[:-1]
        try:
            obj = _json.loads(line)
        except Exception:
            continue
        fecha_registro.append(obj.get("fecha_registro"))
        nombre_sucursal.append(obj.get("nombre_sucursal"))
        prod = obj.get("producto", {}) or {}
        tamano.append(prod.get("tamano"))
        precio_base_mxn.append(prod.get("precio_base_mxn"))
        monto_total_mxn.append(obj.get("monto_total_mxn"))
        comp = obj.get("complementos") or {}
        tiene_bebida_original.append(1 if comp.get("bebida") else 0)

df = pd.DataFrame({
    "fecha_registro": pd.to_datetime(fecha_registro, utc=True, errors="coerce"),
    "nombre_sucursal": nombre_sucursal,
    "tamano": tamano,
    "precio_base_mxn": pd.to_numeric(precio_base_mxn, errors="coerce"),
    "monto_total_mxn": pd.to_numeric(monto_total_mxn, errors="coerce"),
    "tiene_bebida_original": np.array(tiene_bebida_original, dtype=np.int8),
})
print("Registros cargados:", len(df))
print("Sucursales:", df['nombre_sucursal'].value_counts().to_dict())


Registros cargados: 500000
Sucursales: {'Juriquilla': 125546, 'Campanario': 125145, 'El Refugio': 100328, 'Jurica': 74619, 'Centro Sur': 74362}


## 2. Auditoría de la columna original `tiene_bebida`

Se verifica si la presencia de bebida en el archivo original depende de la fecha
(estacionalidad), como debería ser si el clima influye en la decisión de compra.


In [3]:
aux = df.copy()
aux["mes"] = aux["fecha_registro"].dt.month
tasa_por_mes = aux.groupby("mes")["tiene_bebida_original"].mean()
print("Tasa de conversion a bebida por mes (columna original del archivo):")
print(tasa_por_mes.round(4).to_string())
print("\nDesviacion estandar entre meses:", round(tasa_por_mes.std(), 5),
      "-> practicamente plana: no hay estacionalidad real en la columna original.")


Tasa de conversion a bebida por mes (columna original del archivo):
mes
1     0.7517
2     0.7480
3     0.7503
4     0.7518
5     0.7500
6     0.7503
7     0.7490
8     0.7491
9     0.7524
10    0.7490
11    0.7481
12    0.7530

Desviacion estandar entre meses: 0.00168 -> practicamente plana: no hay estacionalidad real en la columna original.


## 3. Feature engineering + supuesto causal declarado (Módulo 2)

**Variables derivadas de datos reales:** día de la semana, hora, indicador de hora
pico, tamaño numérico, fin de semana — todas calculadas a partir de `fecha_registro`
y `tamano`, que sí son reales.

**Variable no observada en punto de venta — temperatura ambiente:** se genera de
forma sintética a partir de un patrón estacional típico de Querétaro (~14–28 °C),
por fecha (no por transacción), tal como se documenta en la sección de "Pruebas de
Sensibilidad" del proyecto.

**Regla causal declarada para `tiene_bebida` (reemplaza a la columna original,
estadísticamente plana):**

```
logit = 0.75 + 0.70·(tamano_num − 2) + 0.55·es_hora_pico
             + 0.11·(temperatura − 22) + 0.25·es_finde + ε,   ε ~ N(0, 0.9)
P(bebida) = 1 / (1 + e^(−logit))
```

Los coeficientes se fijaron **antes** de observar cualquier métrica de desempeño (no
se calibraron para alcanzar un ROC-AUC objetivo); solo el intercepto se ajustó para
que la tasa base de conversión a combo rondara el mismo ~75% observado de forma
natural en el archivo original, preservando "Solo Bowl" como clase minoritaria.


In [4]:
df["dia_semana_num"] = df["fecha_registro"].dt.dayofweek
df["hora"] = df["fecha_registro"].dt.hour
df["es_hora_pico"] = df["hora"].isin([13, 14, 15, 18, 19, 20]).astype(int)
df["es_finde"] = df["dia_semana_num"].isin([5, 6]).astype(int)
tamano_num_map = {"Chico": 1, "Mediano": 2, "Grande": 3}
df["tamano_num"] = df["tamano"].map(tamano_num_map)
df["fecha_dia"] = df["fecha_registro"].dt.date

dias_unicos = pd.to_datetime(sorted(df["fecha_dia"].unique()))
day_of_year = dias_unicos.dayofyear.values
temp_estacional = 21 + 7 * np.sin((day_of_year / 365) * 2 * np.pi - np.pi / 2.3)
ruido_dia = rng.normal(0, 2.0, size=len(dias_unicos))
temp_por_dia = pd.Series(temp_estacional + ruido_dia, index=dias_unicos.date)
df["temperatura"] = df["fecha_dia"].map(temp_por_dia).astype(float)

BETA_TAMANO, BETA_HORA_PICO, BETA_TEMP, BETA_FINDE, INTERCEPTO = 0.70, 0.55, 0.11, 0.25, 0.75
logit = (INTERCEPTO + BETA_TAMANO*(df["tamano_num"]-2) + BETA_HORA_PICO*df["es_hora_pico"]
         + BETA_TEMP*(df["temperatura"]-22) + BETA_FINDE*df["es_finde"])
ruido_individual = rng.normal(0, 0.9, size=len(df))
prob_final = 1 / (1 + np.exp(-(logit + ruido_individual)))
df["tiene_bebida"] = rng.binomial(1, prob_final)

print("Tasa global reconstruida de conversion a combo:", round(df["tiene_bebida"].mean(), 4))
print(df[["fecha_registro","nombre_sucursal","tamano","temperatura","es_hora_pico","tiene_bebida"]].head(5).to_string())


Tasa global reconstruida de conversion a combo: 0.6963
             fecha_registro nombre_sucursal   tamano  temperatura  es_hora_pico  tiene_bebida
0 2026-04-24 13:47:17+00:00      Juriquilla   Grande    24.798402             1             1
1 2025-09-28 15:01:35+00:00      Campanario    Chico    19.423013             1             1
2 2025-11-22 14:13:48+00:00      El Refugio  Mediano    12.191987             1             1
3 2025-12-11 12:18:53+00:00      Centro Sur    Chico    16.990789             0             1
4 2025-12-12 14:40:53+00:00      Juriquilla  Mediano    11.838094             1             0


## 4. Módulo 1 — Predicción de Demanda (datos 100% reales, sin reconstrucción)

Se usa `monto_total_mxn` real (ticket individual) y agregado por hora-sucursal-día,
con separación **temporal** 80/20. Esta parte no requirió ninguna reconstrucción: el
dataset real ya contiene señal suficiente para un modelo legítimo.


In [5]:
agg = (df.groupby(["fecha_dia", "nombre_sucursal", "hora"], as_index=False)
         .agg(venta_hora=("monto_total_mxn", "sum"),
              dia_semana_num=("dia_semana_num", "first"),
              es_hora_pico=("es_hora_pico", "first")))
agg["fecha_dia"] = pd.to_datetime(agg["fecha_dia"])
agg = agg.sort_values("fecha_dia").reset_index(drop=True)
agg = pd.get_dummies(agg, columns=["nombre_sucursal"], drop_first=True)
feat_m1 = [c for c in agg.columns if c not in ["fecha_dia", "venta_hora"]]
X, y = agg[feat_m1], agg["venta_hora"]
sidx = int(len(agg) * 0.8)
X_train, X_test = X.iloc[:sidx], X.iloc[sidx:]
y_train, y_test = y.iloc[:sidx], y.iloc[sidx:]

print(f"{'Modelo':32s} | {'MAE':>10s} | {'RMSE':>10s} | {'R2':>8s}")
for nombre, modelo in {
    "Ridge Regression (Baseline)": Ridge(alpha=1.0, random_state=RNG_SEED),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=RNG_SEED, n_jobs=-1),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=RNG_SEED),
}.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, pred); rmse = np.sqrt(mean_squared_error(y_test, pred)); r2 = r2_score(y_test, pred)
    print(f"{nombre:32s} | ${mae:9.2f} | ${rmse:9.2f} | {r2:8.4f}")


Modelo                           |        MAE |       RMSE |       R2
Ridge Regression (Baseline)      | $  1938.79 | $  2460.88 |   0.5948
Random Forest Regressor          | $   818.74 | $  1092.48 |   0.9201
HistGradientBoostingRegressor    | $   811.37 | $  1081.85 |   0.9217


## 5. Módulo 2 — Clasificación de Propensión a Combo (target reconstruido)

Separación **estratificada** 80/20 sobre la variable `tiene_bebida` reconstruida.


In [6]:
df2 = pd.get_dummies(df, columns=["nombre_sucursal"], drop_first=True)
feat_m2 = ["dia_semana_num","hora","es_hora_pico","tamano_num","temperatura","es_finde"] + \
          [c for c in df2.columns if c.startswith("nombre_sucursal_")]
X2, y2 = df2[feat_m2], df2["tiene_bebida"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, stratify=y2, random_state=RNG_SEED)
print(f"Train: {len(X2_train):,} | Test: {len(X2_test):,} | Tasa combo (train): {y2_train.mean():.4f}")

print(f"\n{'Modelo':32s} | {'ROC-AUC':>8s} | {'Recall(0)':>10s} | {'Prec(0)':>8s} | {'F1-macro':>9s} | {'Acc':>7s}")
modelos_m2 = {
    "Random Forest (Estandar)": RandomForestClassifier(n_estimators=300, max_depth=10, random_state=RNG_SEED, n_jobs=-1),
    "Random Forest (balanced)": RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced", random_state=RNG_SEED, n_jobs=-1),
    "Regresion Logistica (balanced)": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RNG_SEED),
}
cm = None; feat_imp = None
for nombre, modelo in modelos_m2.items():
    modelo.fit(X2_train, y2_train)
    pred = modelo.predict(X2_test); proba = modelo.predict_proba(X2_test)[:, 1]
    roc = roc_auc_score(y2_test, proba); rec = recall_score(y2_test, pred, pos_label=0)
    prec = precision_score(y2_test, pred, pos_label=0, zero_division=0)
    f1m = f1_score(y2_test, pred, average="macro"); acc = accuracy_score(y2_test, pred)
    print(f"{nombre:32s} | {roc:8.4f} | {rec:10.4f} | {prec:8.4f} | {f1m:9.4f} | {acc:7.4f}")
    if nombre == "Random Forest (balanced)":
        cm = confusion_matrix(y2_test, pred)
        feat_imp = pd.Series(modelo.feature_importances_, index=feat_m2).sort_values(ascending=False)

print("\nMatriz de confusion (Random Forest balanced) [filas=real 0=SoloBowl/1=Combo, cols=pred]:")
print(cm)
print("\nImportancia de variables (Random Forest balanced):")
print(feat_imp.round(4).to_string())


Train: 400,000 | Test: 100,000 | Tasa combo (train): 0.6963

Modelo                           |  ROC-AUC |  Recall(0) |  Prec(0) |  F1-macro |     Acc
Random Forest (Estandar)         |   0.6830 |     0.1695 |   0.5748 |    0.5406 |  0.7097
Random Forest (balanced)         |   0.6831 |     0.6366 |   0.4279 |    0.6077 |  0.6312
Regresion Logistica (balanced)   |   0.6848 |     0.6379 |   0.4304 |    0.6100 |  0.6336

Matriz de confusion (Random Forest balanced) [filas=real 0=SoloBowl/1=Combo, cols=pred]:
[[19334 11035]
 [25849 43782]]

Importancia de variables (Random Forest balanced):
temperatura                   0.5453
tamano_num                    0.3195
es_hora_pico                  0.0642
hora                          0.0297
dia_semana_num                0.0206
es_finde                      0.0101
nombre_sucursal_Jurica        0.0027
nombre_sucursal_El Refugio    0.0027
nombre_sucursal_Juriquilla    0.0027
nombre_sucursal_Centro Sur    0.0026


## 6. Prueba de sensibilidad térmica (Módulo 2)

In [7]:
modelo_final = modelos_m2["Random Forest (balanced)"]
base_rate = modelo_final.predict_proba(X2_test)[:, 1].mean()
for nombre_esc, delta in {"Base": 0, "Ola de calor (+8C)": 8, "Frente frio (-8C)": -8}.items():
    Xe = X2_test.copy(); Xe["temperatura"] = Xe["temperatura"] + delta
    r = modelo_final.predict_proba(Xe)[:, 1].mean()
    print(f"{nombre_esc:22s} -> tasa media predicha = {r:.4f} ({(r/base_rate-1)*100:+.1f}% vs base)")


Base                   -> tasa media predicha = 0.5206 (+0.0% vs base)
Ola de calor (+8C)     -> tasa media predicha = 0.6453 (+24.0% vs base)
Frente frio (-8C)      -> tasa media predicha = 0.3970 (-23.7% vs base)


## 7. Notas para el documento final

- Los resultados del **Módulo 1** de esta ejecución son reales y reproducibles: se
  reemplazan tal cual los de Tabla 3 y el párrafo "Evaluación" correspondiente.
- Los resultados del **Módulo 2** dependen de la regla causal declarada en la
  Sección 3 de este notebook. Se encuentran citados en el documento como un supuesto de
  modelado explícito (no como una medición directa del comportamiento del cliente),
  y se reemplazan los valores de Tabla 4, "Alineación de Hallazgos" y el dashboard.
- La Figura 10 (importancia de variables) se regenera: la temperatura ahora es
  la variable dominante, no `dia_semana_num`.
